# Day 2 · Session 3
# 03B Build Your First Agent

**Objective:** Build a simple agent from scratch using Python tools.

## 1. Define Simple Tools
These are normal Python functions. The LLM does not execute them directly; our code does.

In [ ]:
import pandas as pd
import time
import json

def calculator(expression: str):
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Calculation error: {e}"

def search_college_info(query: str):
    data = {
        "mtech_fee": "M.Tech CSE fee is Rs. 50,000 per semester. Duration is 4 semesters.",
        "gate_scholarship": "GATE-qualified students receive 50% tuition scholarship.",
        "attendance": "Students need 75% attendance for final exam eligibility.",
        "nlp": "NLP elective is coordinated by Dr. Kavitha Raman."
    }
    query_lower = query.lower()
    results = []
    for key, value in data.items():
        if key in query_lower or any(word in value.lower() for word in query_lower.split()):
            results.append(value)
    return "\n".join(results) if results else "No matching information found."

print(calculator("50000 * 4 * 0.5"))
print(search_college_info("mtech fee gate scholarship"))

## 2. Tool Registry

In [ ]:
TOOLS = {
    "calculator": calculator,
    "search_college_info": search_college_info
}

list(TOOLS.keys())

## 3. Simple Rule-Based Planner
This simulates how an LLM chooses a tool. Later frameworks do this using function calling.

In [ ]:
def simple_planner(user_question):
    q = user_question.lower()

    if "total" in q and ("fee" in q or "cost" in q or "pay" in q):
        return {
            "tool": "search_college_info",
            "args": {"query": "mtech_fee gate_scholarship"}
        }

    if any(char.isdigit() for char in q) and any(op in q for op in ["+", "-", "*", "/"]):
        return {
            "tool": "calculator",
            "args": {"expression": user_question}
        }

    return {
        "tool": "search_college_info",
        "args": {"query": user_question}
    }

simple_planner("GATE scholar - what will I pay total for M.Tech?")

## 4. Tool Executor

In [ ]:
def execute_tool(tool_name, args):
    if tool_name not in TOOLS:
        return f"Unknown tool: {tool_name}"
    return TOOLS[tool_name](**args)

plan = simple_planner("GATE scholar - what will I pay total for M.Tech?")
result = execute_tool(plan["tool"], plan["args"])

print("Plan:", plan)
print("Tool result:")
print(result)

## 5. First Mini Agent

In [ ]:
def mini_agent(user_question):
    print("User:", user_question)

    plan = simple_planner(user_question)
    print("\nAgent selected tool:", plan["tool"])
    print("Arguments:", plan["args"])

    observation = execute_tool(plan["tool"], plan["args"])
    print("\nObservation:")
    print(observation)

    if "fee is Rs. 50,000" in observation and "50%" in observation:
        total = calculator("50000 * 4 * 0.5")
        final_answer = f"Total M.Tech tuition for a GATE scholar is Rs. {total} over 4 semesters."
    else:
        final_answer = observation

    print("\nFinal Answer:")
    print(final_answer)

    return final_answer

mini_agent("GATE scholar - what will I pay total for M.Tech?")

## 6. Test More Questions

In [ ]:
questions = [
    "Who coordinates NLP?",
    "1250 * 18",
    "What is the attendance rule?",
    "GATE scholar - what will I pay total for M.Tech?"
]

for q in questions:
    print("=" * 80)
    mini_agent(q)
    print()

## Mini Exercise
Add one new tool: `send_notification(name, message)` or `grade_calculator(marks)` and update the planner.